In [45]:
from google.colab import drive
drive.mount('/content/drive', force_remount = True)

Mounted at /content/drive


In [47]:
import os

print(os.path.exists("/content/drive/MyDrive/Datasets/pumpkin"))
print(os.listdir("/content/drive/MyDrive/Datasets/pumpkin"))

True
['Original Dataset']


In [48]:
import os
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.optimizers import Adam

# ---------------------------------------------------------
# PATHS
# ---------------------------------------------------------
DATASET_PATH = "/content/drive/MyDrive/Datasets/pumpkin/Original Dataset"
MODEL_SAVE_PATH = "/content/drive/MyDrive/Datasets/pumpkin/model_pumpkin_disease.keras"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 12


# ---------------------------------------------------------
# Load dataset
# ---------------------------------------------------------
def load_dataset(path):
    filepaths = []
    labels = []

    if not os.path.exists(path):
        print(f"Error: Dataset path does not exist: {path}")
        return pd.DataFrame({"Filepath": filepaths, "Label": labels})

    for class_name in os.listdir(path):
        class_path = os.path.join(path, class_name)

        if os.path.isdir(class_path):
            for img in os.listdir(class_path):
                if img.lower().endswith((".jpg", ".png", ".jpeg")):
                    filepaths.append(os.path.join(class_path, img))
                    labels.append(class_name)

    return pd.DataFrame({"Filepath": filepaths, "Label": labels})


# ---------------------------------------------------------
# Hybrid Model
# ---------------------------------------------------------
def build_model(num_classes):

    effnet = tf.keras.applications.EfficientNetB0(
        include_top=False,
        weights="imagenet",
        pooling="avg",
        input_shape=(224, 224, 3)
    )
    effnet.trainable = False

    convnext = tf.keras.applications.ConvNeXtTiny(
        include_top=False,
        weights="imagenet",
        pooling="avg",
        input_shape=(224, 224, 3)
    )
    convnext.trainable = False

    inputs = tf.keras.Input(shape=(224, 224, 3))

    x1 = effnet(inputs)
    x2 = convnext(inputs)

    x = tf.keras.layers.Concatenate()([x1, x2])
    x = tf.keras.layers.Dense(256, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    x = tf.keras.layers.Dense(64, activation="relu")(x)

    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

    return tf.keras.Model(inputs, outputs)


# ---------------------------------------------------------
# Train
# ---------------------------------------------------------
def train():

    print("GPU Available:", tf.config.list_physical_devices('GPU'))

    print("Loading dataset...")
    df = load_dataset(DATASET_PATH)

    print(f"Found {len(df)} samples in the dataset.") # Added debug print

    if df.empty:
        print("Error: Dataset is empty. Please check DATASET_PATH and its contents.")
        return # Exit if dataset is empty

    # 🔥 Manual Stratified Split
    train_df, val_df = train_test_split(
        df,
        train_size=0.8,
        stratify=df["Label"],
        random_state=42
    )

    # 🔥 Strong Augmentation (important for small dataset)
    train_gen = tf.keras.preprocessing.image.ImageDataGenerator(
        preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
        rotation_range=40,
        zoom_range=0.4,
        width_shift_range=0.2,
        height_shift_range=0.2,
        shear_range=0.3,
        brightness_range=[0.5, 1.5],
        horizontal_flip=True,
        fill_mode="nearest"
    )

    val_gen = tf.keras.preprocessing.image.ImageDataGenerator(
        preprocessing_function=tf.keras.applications.efficientnet.preprocess_input
    )

    train_images = train_gen.flow_from_dataframe(
        train_df,
        x_col="Filepath",
        y_col="Label",
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="categorical"
    )

    val_images = val_gen.flow_from_dataframe(
        val_df,
        x_col="Filepath",
        y_col="Label",
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="categorical"
    )

    print("Classes:", train_images.class_indices)

    model = build_model(len(train_images.class_indices))

    model.compile(
        optimizer=Adam(1e-4),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    print("Training on GPU 🚀 ...")
    history = model.fit(
        train_images,
        validation_data=val_images,
        epochs=EPOCHS
    )

    print("Saving model...")
    model.save(MODEL_SAVE_PATH)

    # Save Accuracy Plot
    plt.figure()
    plt.plot(history.history["accuracy"])
    plt.plot(history.history["val_accuracy"])
    plt.legend(["Train", "Val"])
    plt.title("Accuracy")
    plt.savefig("/content/drive/MyDrive/Datasets/pumpkin/accuracy.png")
    plt.close()

    # Save Loss Plot
    plt.figure()
    plt.plot(history.history["loss"])
    plt.plot(history.history["val_loss"])
    plt.legend(["Train", "Val"])
    plt.title("Loss")
    plt.savefig("/content/drive/MyDrive/Datasets/pumpkin/loss.png")
    plt.close()


train()

GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Loading dataset...
Found 2000 samples in the dataset.
Found 1600 validated image filenames belonging to 5 classes.
Found 400 validated image filenames belonging to 5 classes.
Classes: {'Bacterial Leaf Spot': 0, 'Downy Mildew': 1, 'Healthy Leaf': 2, 'Mosaic Disease': 3, 'Powdery_Mildew': 4}
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
111650432/111650432 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Training on GPU 🚀 ...


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/12
50/50 ━━━━━━━━━━━━━━━━━━━━ 674s 13s/step - accuracy: 0.2831 - loss: 1.6871 - val_accuracy: 0.5200 - val_loss: 1.2461
Epoch 2/12
50/50 ━━━━━━━━━━━━━━━━━━━━ 36s 717ms/step - accuracy: 0.5306 - loss: 1.1988 - val_accuracy: 0.5650 - val_loss: 1.1177
Epoch 3/12
50/50 ━━━━━━━━━━━━━━━━━━━━ 38s 750ms/step - accuracy: 0.6057 - loss: 0.9954 - val_accuracy: 0.6650 - val_loss: 0.9083
Epoch 4/12
50/50 ━━━━━━━━━━━━━━━━━━━━ 37s 736ms/step - accuracy: 0.6285 - loss: 0.9455 - val_accuracy: 0.6300 - val_loss: 0.9126
Epoch 5/12
50/50 ━━━━━━━━━━━━━━━━━━━━ 41s 736ms/step - accuracy: 0.6662 - loss: 0.8674 - val_accuracy: 0.7150 - val_loss: 0.7903
Epoch 6/12
50/50 ━━━━━━━━━━━━━━━━━━━━ 36s 723ms/step - accuracy: 0.6811 - loss: 0.8272 - val_accuracy: 0.7175 - val_loss: 0.7754
Epoch 7/12
50/50 ━━━━━━━━━━━━━━━━━━━━ 37s 742ms/step - accuracy: 0.7410 - loss: 0.7368 - val_accuracy: 0.7175 - val_loss: 0.7318
Epoch 8/12
50/50 ━━━━━━━━━━━━━━━━━━━━ 37s 742ms/step - accuracy: 0.7439 - loss: 0.6936 - val_accur